# Predicting Smartphone Addiction — clean Rank-Gauss stack

Упрощённая версия исходного notebook без исследовательских экспериментов.

Цель этой версии — оставить только путь, который непосредственно нужен для получения той же модели и того же submission:

1. загрузить `train.csv` и `test.csv`;
2. собрать опубликованные OOF/test-предсказания базовых моделей;
3. убрать почти полные дубликаты и self-referential stack-модели;
4. преобразовать предсказания через percentile rank → Rank-Gauss;
5. обучить cross-fitted logistic regression как meta-model;
6. получить честный OOF ROC-AUC;
7. сделать тот же финальный blend с публичными submission;
8. сохранить итоговый `submission.csv`.

При тех же Kaggle Inputs и версиях библиотек ожидаемый результат исходного notebook:

- **stack OOF ROC-AUC ≈ 0.970167**
- финальный public LB исходного решения: **≈ 0.97130**

> Важно: это не обучение одной модели с нуля на признаках соревнования. Это **stacking** большого набора уже опубликованных OOF/test-предсказаний других моделей. Поэтому notebook требует тех же внешних Kaggle datasets, что и оригинал.


## 1. Импорты и конфигурация

Все внешние файлы Kaggle обычно находятся в `/kaggle/input`.

Чтобы сохранить тот же результат, мы не меняем:

- порядок строк `train.csv`;
- схему `StratifiedKFold`;
- порядок моделей после сортировки;
- порог удаления почти одинаковых моделей;
- Rank-Gauss преобразование;
- параметры logistic regression;
- вес финального blend.


In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
from scipy.stats import norm, rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler


N_TRAIN = 691_369
N_TEST = 296_302
INPUT = os.environ.get("S6E8_INPUT", "/kaggle/input")


def find_file(filename, must_contain=None):
    """Ищет файл по всему Kaggle input.

    must_contain позволяет выбрать нужный dataset, если файлов
    с одинаковым именем несколько.
    """
    hits = glob.glob(f"{INPUT}/**/{filename}", recursive=True)

    if must_contain:
        hits = [
            path
            for path in hits
            if must_contain.lower() in path.lower()
        ]

    if not hits:
        raise FileNotFoundError(
            f"{filename!r} (containing {must_contain!r}) "
            f"not found under {INPUT}"
        )

    return sorted(hits, key=len)[0]


def find_dir(folder):
    """Ищет директорию подключённого Kaggle dataset."""
    hits = [
        path
        for path in glob.glob(f"{INPUT}/**/{folder}", recursive=True)
        if os.path.isdir(path)
    ]

    return sorted(hits, key=len)[0] if hits else None


## 2. Данные и схема кросс-валидации

OOF-массивы из подключённых datasets были построены на одной и той же схеме:

```python
StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
```

Это критично: `.npy`-массивы не содержат `id`, поэтому соответствие идёт **по позиции строки**.

Нельзя сортировать или переиндексировать `train` до работы со stack.


In [ ]:
train = pd.read_csv(
    find_file("train.csv", "playground-series-s6e8")
)
test = pd.read_csv(
    find_file("test.csv", "playground-series-s6e8")
)

assert len(train) == N_TRAIN
assert len(test) == N_TEST

y = train["addicted_label"].to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

folds = list(
    cv.split(np.zeros(N_TRAIN), y)
)

print(
    f"{len(train):,} train rows, "
    f"{len(test):,} test rows, "
    f"positive rate {y.mean():.4f}"
)


## 3. Собираем библиотеку OOF/test-предсказаний

Для stacking нужны две версии предсказаний каждой базовой модели:

- `OOF` — предсказания на train-строках, которые модель не видела при обучении;
- `test` — предсказания той же модели для competition test.

Именно OOF позволяет обучать meta-model без прямой утечки target.

Разные авторы публиковали массивы под немного разными именами, поэтому функция ниже ищет несколько вариантов имени test-файла.


In [ ]:
def load_pairs(
    root,
    prefix,
    test_prefixes=("test_", "testpred_", "tep_"),
):
    """Загружает пары oof_<name>.npy + test_<name>.npy."""
    result = {}

    pattern = os.path.join(root, "**", "oof_*.npy")

    for oof_path in glob.glob(pattern, recursive=True):
        name = os.path.basename(oof_path)[4:-4]

        candidates = [
            os.path.join(
                os.path.dirname(oof_path),
                test_prefix + name + ".npy",
            )
            for test_prefix in test_prefixes
        ]

        test_path = next(
            (
                candidate
                for candidate in candidates
                if os.path.exists(candidate)
            ),
            None,
        )

        if test_path is None:
            continue

        oof = np.load(oof_path).astype(np.float64)
        test_pred = np.load(test_path).astype(np.float64)

        valid_shapes = (
            oof.shape == (N_TRAIN,)
            and test_pred.shape == (N_TEST,)
        )
        finite_values = (
            np.isfinite(oof).all()
            and np.isfinite(test_pred).all()
        )

        if valid_shapes and finite_values:
            result[prefix + name] = (oof, test_pred)

    return result


SOURCES = [
    ("s6e8-oof-library-47-models", "sz_"),
    ("s6e8-oof-library-11-members", "nn_"),
    ("s6e8-mask-augmented-oof-library", "ma_"),
    ("s6e8-full-best-blend-npy", "tam_"),
    ("s6e8-adarsh-oof-library", "a_"),
    ("s6e8-golem-oof-library", "golem_"),
    ("s6e8-fm-lattice-blend-members", "fm_"),
    ("s6e8-150-fusion-local-members", "hb_"),
    ("s6e8-catstrall-member", "x_"),
    ("s6e8-catstr-aug16", "mk_"),
]

members = {}

for folder, prefix in SOURCES:
    root = find_dir(folder)
    loaded = load_pairs(root, prefix) if root else {}

    members.update(loaded)

    suffix = "" if root else "   NOT ATTACHED"
    print(f"{folder:36s} {len(loaded):3d}{suffix}")


### Дополнительные форматы опубликованных предсказаний

Две библиотеки хранят предсказания не как отдельные пары `.npy`:

- одна — в parquet-таблицах;
- другая — как матрицу `(n_rows, n_models)`.

Добавляем их в тот же словарь `members`.


In [ ]:
# Библиотека в parquet: один столбец = одна модель
bolt = find_dir("s6e8-oof-prediction-library") or ""

if bolt and os.path.exists(f"{bolt}/oof_predictions.parquet"):
    oof_df = pd.read_parquet(
        f"{bolt}/oof_predictions.parquet"
    )
    test_df = pd.read_parquet(
        f"{bolt}/test_predictions.parquet"
    )

    for column in oof_df.columns:
        if column != "id" and column in test_df:
            members[f"bolt_{column}"] = (
                oof_df[column].to_numpy(float),
                test_df[column].to_numpy(float),
            )


# 50 слабых моделей хранятся одной двумерной матрицей
weak = find_dir("s6e8-50-weakest-oof-models") or ""

if weak and os.path.exists(f"{weak}/oof.npy"):
    weak_oof = np.load(
        f"{weak}/oof.npy",
        mmap_mode="r",
    )
    weak_test = np.load(
        f"{weak}/test.npy",
        mmap_mode="r",
    )

    for j in range(weak_oof.shape[1]):
        members[f"weak_{j:02d}"] = (
            np.asarray(weak_oof[:, j], float),
            np.asarray(weak_test[:, j], float),
        )


# Сортировка важна для воспроизводимости порядка столбцов.
names = sorted(members)

OOF = np.column_stack(
    [members[name][0] for name in names]
)
TST = np.column_stack(
    [members[name][1] for name in names]
)

member_auc = pd.Series(
    [
        roc_auc_score(y, OOF[:, j])
        for j in range(OOF.shape[1])
    ],
    index=names,
)

print(
    f"{len(names)} members, "
    f"OOF AUC from {member_auc.min():.5f} "
    f"to {member_auc.max():.5f}"
)


## 4. Удаляем почти одинаковые модели

Если два столбца дают практически одинаковое ранжирование объектов, stacking будет фактически учитывать одну и ту же модель несколько раз.

Сначала превращаем каждое предсказание в percentile rank, затем считаем корреляцию.

Если rank-correlation выше `0.9995`, оставляем из пары модель с более высоким OOF ROC-AUC.


In [ ]:
def pct_rank(values):
    """Percentile rank в интервале (0, 1)."""
    return (
        rankdata(values) - 0.5
    ) / len(values)


R = np.column_stack(
    [
        pct_rank(OOF[:, j])
        for j in range(OOF.shape[1])
    ]
).astype(np.float32)

Rt = np.column_stack(
    [
        pct_rank(TST[:, j])
        for j in range(TST.shape[1])
    ]
).astype(np.float32)


# После стандартизации скалярное произведение даёт
# матрицу корреляций и экономит память.
Z = (
    R - R.mean(axis=0)
) / (
    R.std(axis=0) + 1e-12
)

corr = (Z.T @ Z) / len(Z)
del Z


drop = set()

for i in range(len(names)):
    if names[i] in drop:
        continue

    for j in range(i + 1, len(names)):
        if names[j] in drop:
            continue

        if corr[i, j] <= 0.9995:
            continue

        if member_auc[names[i]] >= member_auc[names[j]]:
            drop.add(names[j])
        else:
            drop.add(names[i])


print(
    f"{len(drop)} near-duplicate members dropped "
    "at rank correlation > 0.9995"
)


## 5. Убираем self-referential members

Некоторые опубликованные predictors сами уже являются blends/stacks, обученными на частично том же пуле.

Их OOF может выглядеть лучше не потому, что они несут новый независимый сигнал, а потому что они повторно используют уже известные meta-model предсказания.

Чтобы meta-model не переоценивал такие столбцы, исходное решение исключает их по имени.


In [ ]:
SELF_REFERENTIAL = re.compile(
    r"^(naji|sz_naji|v13_anchor|hb_candidate)"
)

keep = [
    i
    for i, name in enumerate(names)
    if (
        name not in drop
        and not SELF_REFERENTIAL.match(name)
    )
]

names = [names[i] for i in keep]
R = R[:, keep]
Rt = Rt[:, keep]

print(f"{len(names)} members kept")


## 6. Rank-Gauss преобразование

Разные модели могут выдавать вероятности в очень разных шкалах.

Для ROC-AUC главным является **порядок объектов**, а не абсолютная калибровка вероятностей.

Поэтому:

1. сначала каждое предсказание переводится в percentile rank;
2. затем percentile rank пропускается через обратную CDF стандартного нормального распределения.

Так meta-model работает с сопоставимыми по масштабу признаками, сохраняя ранжирование каждой базовой модели.


In [ ]:
G = norm.ppf(
    np.clip(R, 1e-7, 1 - 1e-7)
).astype(np.float32)

Gt = norm.ppf(
    np.clip(Rt, 1e-7, 1 - 1e-7)
).astype(np.float32)

print(
    "meta feature matrices:",
    G.shape,
    Gt.shape,
)


## 7. Cross-fitted logistic regression

Meta-model — обычная Logistic Regression.

Но её тоже нужно валидировать честно:

- на каждом fold веса stack обучаются только на `fit_idx`;
- прогноз для `val_idx` делается моделью, которая эти строки не использовала;
- после пяти folds получаем полный OOF-вектор и считаем ROC-AUC.

`StandardScaler` обучается отдельно внутри каждого fold — это часть pipeline meta-model и не должна видеть validation fold.


In [ ]:
def fit_logistic(
    X_fit,
    y_fit,
    X_pred,
    C=1.0,
):
    scaler = StandardScaler().fit(X_fit)

    model = LogisticRegression(
        C=C,
        max_iter=3000,
        solver="lbfgs",
        tol=1e-5,
    )

    model.fit(
        scaler.transform(X_fit),
        y_fit,
    )

    assert int(np.max(model.n_iter_)) < 3000, (
        "meta-model did not converge"
    )

    return model.predict_proba(
        scaler.transform(X_pred)
    )[:, 1]


oof_meta = np.zeros(N_TRAIN)

for fold_number, (fit_idx, val_idx) in enumerate(folds):
    oof_meta[val_idx] = fit_logistic(
        G[fit_idx],
        y[fit_idx],
        G[val_idx],
    )

    fold_auc = roc_auc_score(
        y[val_idx],
        oof_meta[val_idx],
    )

    print(
        f"fold {fold_number}: "
        f"ROC-AUC = {fold_auc:.6f}"
    )


stack_auc = roc_auc_score(y, oof_meta)
print(f"\nstack OOF AUC = {stack_auc:.6f}")


# После честной OOF-оценки обучаем meta-model
# уже на всех train-строках и прогнозируем Kaggle test.
test_meta = fit_logistic(
    G,
    y,
    Gt,
)


## 8. Финальный blend

Исходный notebook получил лучший public leaderboard score не чистым stack, а blend:

- `35%` — наш stack;
- `65%` — сильный публичный blend из dataset с `vault` в пути.

Сначала два публичных submission смешиваются в пропорции `2.9 : 0.1`, затем ещё раз переводятся в rank.

Чтобы итоговый файл был **таким же по логике**, сохраняем тот же вес `W = 0.35`.

Также создаём второй файл — `submission_honest_stack.csv` — только из нашего stack без leaderboard-blend.


In [ ]:
public_1 = pd.read_csv(
    find_file("submission.csv", "vault")
)["addicted_label"].to_numpy()

public_2 = pd.read_csv(
    find_file("submission (1).csv", "vault")
)["addicted_label"].to_numpy()


public = pct_rank(
    (2.9 * public_1 + 0.1 * public_2) / 3.0
)

W = 0.35

final = pct_rank(
    W * pct_rank(test_meta)
    + (1 - W) * public
)


rank_corr = np.corrcoef(
    pct_rank(test_meta),
    public,
)[0, 1]

print(
    "rank correlation, stack vs public submission: "
    f"{rank_corr:.5f}"
)


## 9. Сохраняем submissions

Основной файл:

- `submission.csv` — тот же aggressive blend, который использовался в исходном решении.

Дополнительный:

- `submission_honest_stack.csv` — чистый stack, без смешивания с public submission.


In [ ]:
sub_honest = pd.DataFrame(
    {
        "id": test["id"],
        "addicted_label": pct_rank(test_meta),
    }
)

sub_honest.to_csv(
    "submission_honest_stack.csv",
    index=False,
)


submission = pd.DataFrame(
    {
        "id": test["id"],
        "addicted_label": final,
    }
)

assert len(submission) == N_TEST
assert submission["id"].is_unique

submission.to_csv(
    "submission.csv",
    index=False,
)

print(
    "saved: submission.csv "
    "+ submission_honest_stack.csv"
)


## Что было удалено из исходного notebook

Из этой версии специально убраны эксперименты, которые **не меняют финальную модель**:

- дополнительный LightGBM на exact-value target encoding;
- проверка, принять ли этот LightGBM в stack;
- split-half анализ exact-value эффектов;
- таблица всех неудачных/удачных экспериментов;
- симуляция шума Public Leaderboard;
- анализ прошлых Season 6 public/private leaderboard;
- длинное обсуждение альтернативных blend weights.

Они полезны как исследование, но не нужны для воспроизводимого пути:

`inputs → OOF library → filtering → Rank-Gauss → logistic stack → blend → submission`.

Если все исходные Kaggle datasets подключены, эта версия должна воспроизводить ту же вычислительную цепочку, которая дала исходный submission.
